# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishamerja1903/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [11]:
# Section 1 - Ranked actions + reason codes

import pandas as pd
import numpy as np

# Load the public-safe starter dataset
url = "https://raw.githubusercontent.com/krishamerja1903/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Dataset loaded:", len(df), "rows")

# Keep rows with the signals required for the action queue
queue = df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "days_since_last_update",
        "trend_direction",
        "trend_pct"
    ]
].copy()

# Candidate pages: observed downward trend with meaningful search exposure
queue = queue[
    (queue["trend_direction"] == "down") &
    (queue["impressions_90d"] >= 1000)
].copy()

# Simple transparent decision-support score
queue["priority_score"] = (
    np.log1p(queue["impressions_90d"]) *
    np.abs(queue["trend_pct"].fillna(0))
)

queue = queue.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

# Human-readable action bands
queue["recommended_action"] = "MONITOR"
queue.loc[queue.index < 190, "recommended_action"] = "REVIEW"
queue.loc[queue.index < 16, "recommended_action"] = "PRIORITY_REVIEW"

# Reason codes
def reason_code(row):
    reasons = []

    if row["trend_direction"] == "down":
        reasons.append("DOWNWARD_TREND")

    if row["impressions_90d"] >= 10000:
        reasons.append("HIGH_VISIBILITY")

    if row["days_since_last_update"] > 90:
        reasons.append("OLDER_CONTENT")

    return " | ".join(reasons)

queue["reason_code"] = queue.apply(reason_code, axis=1)

print("\nACTION QUEUE")
print("Queue rows:", len(queue))
print("\nRecommended actions:")
print(queue["recommended_action"].value_counts())

print("\nTop 10 review candidates:")
print(
    queue[
        [
            "content_id",
            "priority_score",
            "recommended_action",
            "reason_code"
        ]
    ].head(10).to_string(index=False)
)

print(
    "\nThis queue is decision-support only. "
    "A human reviewer decides whether any content action should be taken."
)

Dataset loaded: 30000 rows

ACTION QUEUE
Queue rows: 8031

Recommended actions:
recommended_action
MONITOR            7841
REVIEW              174
PRIORITY_REVIEW      16
Name: count, dtype: int64

Top 10 review candidates:
          content_id  priority_score recommended_action                                      reason_code
content_66b4046cc144     1097.458409    PRIORITY_REVIEW                 DOWNWARD_TREND | HIGH_VISIBILITY
content_ec66c58d9826     1059.148690    PRIORITY_REVIEW                 DOWNWARD_TREND | HIGH_VISIBILITY
content_33da44cb09c9     1035.559059    PRIORITY_REVIEW                 DOWNWARD_TREND | HIGH_VISIBILITY
content_20e876d26019     1034.119019    PRIORITY_REVIEW                 DOWNWARD_TREND | HIGH_VISIBILITY
content_7a6df559322d     1011.773045    PRIORITY_REVIEW DOWNWARD_TREND | HIGH_VISIBILITY | OLDER_CONTENT
content_3437133c7ccf     1000.525107    PRIORITY_REVIEW                 DOWNWARD_TREND | HIGH_VISIBILITY
content_551fe371f51b      980.566766    P

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [16]:
# Section 2 - Intended use and limits

print("INTENDED USE AND LIMITS")

print("""
Intended use:
- This queue helps content teams prioritize pages for human review.
- Rankings use observed historical search and content signals.
- Recommended actions are decision-support, not automatic decisions.

Limits:
- The analysis is observational and does not establish causality.
- A high priority score does not prove that refreshing a page will improve performance.
- Historical patterns may change over time.
- Human review is required before any content change is made.
""")

print("Queue available for review:", len(queue), "pages")

INTENDED USE AND LIMITS

Intended use:
- This queue helps content teams prioritize pages for human review.
- Rankings use observed historical search and content signals.
- Recommended actions are decision-support, not automatic decisions.

Limits:
- The analysis is observational and does not establish causality.
- A high priority score does not prove that refreshing a page will improve performance.
- Historical patterns may change over time.
- Human review is required before any content change is made.

Queue available for review: 8031 pages


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [17]:
# Section 3 - Human review + the no-go list

print("HUMAN REVIEW CHECKLIST")

print("""
Before acting, a reviewer should check:
1. Whether the observed decline is meaningful and persistent.
2. Whether the page still matches its intended search intent.
3. Whether the content is genuinely outdated or incomplete.
4. Whether seasonality or unusual traffic patterns could explain the change.
5. Whether updating the page is appropriate for the user and business context.

NO-GO LIST:
- Do not automatically rewrite, delete, or publish content.
- Do not treat the priority score as causal proof.
- Do not expose client identities, URLs, or private queries.
- Do not take action without human review.
""")

# Privacy sanity check
private_fields = {"client_name", "domain", "url", "query"}
used_fields = set(queue.columns)

print("Private fields used:", sorted(private_fields.intersection(used_fields)))
print("Human review required: YES")

HUMAN REVIEW CHECKLIST

Before acting, a reviewer should check:
1. Whether the observed decline is meaningful and persistent.
2. Whether the page still matches its intended search intent.
3. Whether the content is genuinely outdated or incomplete.
4. Whether seasonality or unusual traffic patterns could explain the change.
5. Whether updating the page is appropriate for the user and business context.

NO-GO LIST:
- Do not automatically rewrite, delete, or publish content.
- Do not treat the priority score as causal proof.
- Do not expose client identities, URLs, or private queries.
- Do not take action without human review.

Private fields used: []
Human review required: YES


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [18]:
# Section 4 - Monitoring / retrain triggers

print("MONITORING / RETRAIN TRIGGERS")

print("""
Review or rebuild the recommendation logic when:
- Input feature distributions shift materially from the reference data.
- The share of pages receiving each action changes substantially.
- Validation performance declines on newer data.
- Search or content behavior changes enough that historical relationships no longer hold.
- New data definitions or pipeline changes alter the meaning of the features.

The queue should be treated as stale when its assumptions no longer match current data.
""")

action_share = (
    queue["recommended_action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Current action distribution (%):")
print(action_share)

MONITORING / RETRAIN TRIGGERS

Review or rebuild the recommendation logic when:
- Input feature distributions shift materially from the reference data.
- The share of pages receiving each action changes substantially.
- Validation performance declines on newer data.
- Search or content behavior changes enough that historical relationships no longer hold.
- New data definitions or pipeline changes alter the meaning of the features.

The queue should be treated as stale when its assumptions no longer match current data.

Current action distribution (%):
recommended_action
MONITOR            97.63
REVIEW              2.17
PRIORITY_REVIEW     0.20
Name: proportion, dtype: float64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
# Section 5 - Exports for the paper

import os
import json

os.makedirs("work/outputs", exist_ok=True)

queue_path = "work/outputs/action_queue.csv"
metrics_path = "work/outputs/action_playbook_metrics.json"

queue.to_csv(queue_path, index=False)

metrics = {
    "queue_rows": int(len(queue)),
    "priority_review": int((queue["recommended_action"] == "PRIORITY_REVIEW").sum()),
    "review": int((queue["recommended_action"] == "REVIEW").sum()),
    "monitor": int((queue["recommended_action"] == "MONITOR").sum()),
    "framing": "observational decision-support; human review required"
}

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("EXPORT CHECK")
print("OK -", queue_path)
print("OK -", metrics_path)

print("\nExported metrics:")
print(json.dumps(metrics, indent=2))

EXPORT CHECK
OK - work/outputs/action_queue.csv
OK - work/outputs/action_playbook_metrics.json

Exported metrics:
{
  "queue_rows": 8031,
  "priority_review": 16,
  "review": 174,
  "monitor": 7841,
  "framing": "observational decision-support; human review required"
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.